# Processing Elasticsearch Bulk API Requests with Logstash

This notebook runs the whole tutorial end to end:

1. Download and start Elasticsearch (as a non-root user, since Elasticsearch refuses to run as root).
2. Download and start Logstash with the reference pipeline that pairs bulk action and source lines.
3. Send a sample Bulk API request to the Logstash http input.
4. Verify the four operations (index, create, update, delete) resolved correctly in Elasticsearch.
5. Inspect live pipeline analytics with tuistash, an open source terminal UI for Logstash.

**About the setup.** On a laptop the easiest way to run Elasticsearch and Kibana locally is `start-local`, a one line Docker install:

```
curl -fsSL https://elastic.co/start-local | sh
```

Google Colab has no Docker daemon, so this notebook instead downloads the Elasticsearch and Logstash tarballs and runs them as background processes. It is the same stack, no Docker needed. Security and TLS are turned off here for convenience only. Never do that outside a throwaway sandbox.

Runtime notes: the first run downloads roughly 1 GB of archives, so give it a few minutes. A standard Colab CPU runtime with the default memory is enough.


## Step 0. Configuration and helpers

In [5]:
import os, time, socket, subprocess, json, urllib.request

# Change this to any published Elastic Stack version you want to test.
STACK_VERSION = "9.3.0"
ARCH = "linux-x86_64"
HTTP_PORT = 9700  # Colab often occupies 8080; use a free port for the http input

WORK = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(WORK)
ES_HOME = f"{WORK}/elasticsearch-{STACK_VERSION}"
LS_HOME = f"{WORK}/logstash-{STACK_VERSION}"
print("Working dir:", WORK)
print("Stack version:", STACK_VERSION)


def wait_http(url, timeout=240, name="service"):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                if r.status == 200:
                    print(name, "is up:", url)
                    return True
        except Exception:
            pass
        time.sleep(3)
    print(name, "did not respond within", timeout, "seconds")
    return False


def wait_port(host, port, timeout=180, name="port"):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket() as s:
            s.settimeout(2)
            try:
                s.connect((host, port))
                print(name, "listening on", f"{host}:{port}")
                return True
            except Exception:
                pass
        time.sleep(3)
    print(name, "not listening after", timeout, "seconds")
    return False


Working dir: /content
Stack version: 9.3.0


## Step 1. Download and start Elasticsearch

Elasticsearch refuses to start as root, so the notebook creates a dedicated `esuser`, hands it the install directory, and launches the process under that account. Security and the geoip downloader are disabled to keep the local run self contained.

In [6]:
es_tar = f"elasticsearch-{STACK_VERSION}-{ARCH}.tar.gz"
es_url = f"https://artifacts.elastic.co/downloads/elasticsearch/{es_tar}"

if not os.path.isdir(ES_HOME):
    if not os.path.exists(es_tar):
        print("Downloading", es_url)
        urllib.request.urlretrieve(es_url, es_tar)
    print("Extracting", es_tar)
    subprocess.run(["tar", "-xzf", es_tar], check=True)
print("ES_HOME:", ES_HOME)


ES_HOME: /content/elasticsearch-9.3.0


In [7]:
# Minimal, local-only configuration
es_yml = """cluster.name: colab-cluster
discovery.type: single-node
network.host: 127.0.0.1
http.port: 9200
xpack.security.enabled: false
xpack.ml.enabled: false
ingest.geoip.downloader.enabled: false
"""
with open(f"{ES_HOME}/config/elasticsearch.yml", "w") as f:
    f.write(es_yml)

# Keep the heap small so it fits the Colab runtime
os.makedirs(f"{ES_HOME}/config/jvm.options.d", exist_ok=True)
with open(f"{ES_HOME}/config/jvm.options.d/heap.options", "w") as f:
    f.write("-Xms1g\n-Xmx1g\n")

# Create a non-root user and give it ownership, then start Elasticsearch under it.
# runuser is standard on Ubuntu/Colab; if it is missing, swap in: su esuser -c "..."
subprocess.run("id esuser >/dev/null 2>&1 || useradd -m esuser", shell=True)
subprocess.run(["chown", "-R", "esuser:esuser", ES_HOME], check=True)

es_proc = subprocess.Popen(
    ["runuser", "-u", "esuser", "--", "bash", "-c", f"cd {ES_HOME} && exec ./bin/elasticsearch"],
    stdout=open(f"{WORK}/es.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
)
print("Elasticsearch starting, PID:", es_proc.pid)

if not wait_http("http://127.0.0.1:9200", 300, "Elasticsearch"):
    print("---- last lines of es.log ----")
    print(subprocess.run(["tail", "-n", "40", f"{WORK}/es.log"], capture_output=True, text=True).stdout)


Elasticsearch starting, PID: 24140
Elasticsearch is up: http://127.0.0.1:9200


In [8]:
# Confirm the cluster is healthy
import requests
print(requests.get("http://127.0.0.1:9200").json())
print(requests.get("http://127.0.0.1:9200/_cluster/health?pretty").text)


{'name': '82fb42c5af44', 'cluster_name': 'colab-cluster', 'cluster_uuid': 'qTbOj3uFRga_0ogx4dJk_A', 'version': {'number': '9.3.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '17b451d8979a29e31935fe1eb901310350b30e62', 'build_date': '2026-01-29T10:05:46.708397977Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}
{
  "cluster_name" : "colab-cluster",
  "status" : "green",
  "timed_out" : false,
  "number_of_nodes" : 1,
  "number_of_data_nodes" : 1,
  "active_primary_shards" : 0,
  "active_shards" : 0,
  "relocating_shards" : 0,
  "initializing_shards" : 0,
  "unassigned_shards" : 0,
  "unassigned_primary_shards" : 0,
  "delayed_unassigned_shards" : 0,
  "number_of_pending_tasks" : 0,
  "number_of_in_flight_fetch" : 0,
  "task_max_waiting_in_queue_millis" : 0,
  "active_shards_percent_as_number" : 100.0
}



## Step 2. Download and start Logstash

Logstash 9.x refuses to run as superuser, and Colab runs as root, so the notebook sets `allow_superuser: true` in `logstash.yml`. The reference pipeline is written to `config/bulk.conf` with `pipeline.workers: 1` and `pipeline.ordered: true` so the aggregate filter pairs action and source lines deterministically. Colab also tends to occupy port 8080, so the http input listens on `HTTP_PORT` (9700 by default).

In [9]:
ls_tar = f"logstash-{STACK_VERSION}-{ARCH}.tar.gz"
ls_url = f"https://artifacts.elastic.co/downloads/logstash/{ls_tar}"

if not os.path.isdir(LS_HOME):
    if not os.path.exists(ls_tar):
        print("Downloading", ls_url)
        urllib.request.urlretrieve(ls_url, ls_tar)
    print("Extracting", ls_tar)
    subprocess.run(["tar", "-xzf", ls_tar], check=True)
print("LS_HOME:", LS_HOME)


Extracting logstash-9.3.0-linux-x86_64.tar.gz
LS_HOME: /content/logstash-9.3.0


In [10]:
pipeline_conf = r"""input {
  http {
    port => 8080
    additional_codecs => {
      "application/x-ndjson" => "json_lines"
    }
  }
}

filter {
  # Request correlation (X-Request-Id)
  ruby {
    code => '
      headers = event.get("[@metadata][input][http][request][headers]") || {}
      reqid =
        headers["x_request_id"] ||
        headers["x-request-id"]
      if reqid
        event.set("bulk_reqid", reqid)
      else
        event.tag("missing_reqid")
      end
    '
  }

  # Bulk action detection (order independent)
  ruby {
    code => '
      action = nil
      ["index","create","update","delete"].each do |k|
        v = event.get(k)
        if v.is_a?(Hash)
          action = k
          break
        end
      end
      if action
        event.set("is_bulk_action_line", true)
        event.set("bulk_action", action)
        meta = event.get(action)
        event.set("bulk_index", meta["_index"])
        event.set("bulk_id", meta["_id"])
      else
        event.set("is_bulk_action_line", false)
      end
    '
  }

  # Action lines
  if [is_bulk_action_line] {
    if [bulk_action] in ["index","create","update"] {
      aggregate {
        task_id => "%{bulk_reqid}"
        code => '
          map["queue"] ||= []
          map["queue"] << {
            "action" => event.get("bulk_action"),
            "_index" => event.get("bulk_index"),
            "_id"    => event.get("bulk_id")
          }
        '
      }
      drop { }
    } else if [bulk_action] == "delete" {
      mutate {
        add_field => {
          "[@metadata][action]" => "%{bulk_action}"
          "[@metadata][_index]" => "%{bulk_index}"
          "[@metadata][_id]"    => "%{bulk_id}"
        }
      }
    }

  # Document (source) lines
  } else {
    aggregate {
      task_id => "%{bulk_reqid}"
      code => '
        q = map["queue"] || []
        meta = q.shift
        map["queue"] = q
        if meta
          event.set("[@metadata][action]", meta["action"])
          event.set("[@metadata][_index]", meta["_index"])
          event.set("[@metadata][_id]",    meta["_id"])
        else
          event.tag("bulk_bad_event")
        end
      '
    }

    if [@metadata][action] == "update" and [doc] {
      ruby {
        code => '
          d = event.get("doc")
          if d.is_a?(Hash)
            event.remove("doc")
            d.each { |k,v| event.set(k,v) }
          end
        '
      }
    }
  }

  # Safety guard
  if ![@metadata][action] {
    mutate { add_tag => ["bulk_bad_event"] }
  }

  # Keep only the client's source document. Strip Logstash event decorations
  # and the pipeline's internal working fields before indexing.
  mutate {
    remove_field => [
      "is_bulk_action_line", "bulk_action", "bulk_index", "bulk_id", "bulk_reqid",
      "host", "http", "url", "user_agent", "event", "@version", "@timestamp"
    ]
  }
}

output {
  if "bulk_bad_event" in [tags] {
    stdout { codec => rubydebug { metadata => true } }
  } else {
    elasticsearch {
      hosts       => ["http://127.0.0.1:9200"]
      action      => "%{[@metadata][action]}"
      index       => "%{[@metadata][_index]}"
      document_id => "%{[@metadata][_id]}"
    }
  }
}
"""

pipeline_conf = pipeline_conf.replace("port => 8080", f"port => {HTTP_PORT}")

with open(f"{LS_HOME}/config/bulk.conf", "w") as f:
    f.write(pipeline_conf)

ls_yml = """pipeline.workers: 1
pipeline.ordered: true
api.http.host: 127.0.0.1
api.http.port: 9600
allow_superuser: true
"""
with open(f"{LS_HOME}/config/logstash.yml", "w") as f:
    f.write(ls_yml)

print("Wrote pipeline to", f"{LS_HOME}/config/bulk.conf")
print(pipeline_conf)

Wrote pipeline to /content/logstash-9.3.0/config/bulk.conf
input {
  http {
    port => 9700
    additional_codecs => {
      "application/x-ndjson" => "json_lines"
    }
  }
}

filter {
  # Request correlation (X-Request-Id)
  ruby {
    code => '
      headers = event.get("[@metadata][input][http][request][headers]") || {}
      reqid =
        headers["x_request_id"] ||
        headers["x-request-id"]
      if reqid
        event.set("bulk_reqid", reqid)
      else
        event.tag("missing_reqid")
      end
    '
  }

  # Bulk action detection (order independent)
  ruby {
    code => '
      action = nil
      ["index","create","update","delete"].each do |k|
        v = event.get(k)
        if v.is_a?(Hash)
          action = k
          break
        end
      end
      if action
        event.set("is_bulk_action_line", true)
        event.set("bulk_action", action)
        meta = event.get(action)
        event.set("bulk_index", meta["_index"])
        event.set("bulk_id", meta[

In [ ]:
# Start Logstash (clean start; pkill clears any earlier instance, does not touch Elasticsearch)
subprocess.run("pkill -f 'logstash' 2>/dev/null; sleep 3", shell=True)

ls_env = dict(os.environ, LS_JAVA_OPTS="-Xms512m -Xmx512m")
ls_proc = subprocess.Popen(
    ["bash", "-c", f"cd {LS_HOME} && exec ./bin/logstash -f config/bulk.conf"],
    stdout=open(f"{WORK}/logstash.log", "w"), stderr=subprocess.STDOUT,
    start_new_session=True, env=ls_env,
)
print("Logstash starting, PID:", ls_proc.pid, "(startup takes ~40-60s)")

# Monitoring API on 9600 comes up first, then the http input binds on HTTP_PORT
if not wait_http("http://127.0.0.1:9600", 300, "Logstash API"):
    print(subprocess.run(["tail", "-n", "40", f"{WORK}/logstash.log"], capture_output=True, text=True).stdout)
wait_port("127.0.0.1", HTTP_PORT, 180, "Logstash http input")
time.sleep(5)  # small cushion so the input is fully ready

## Step 3. Send a Bulk API request

The sample payload runs one of each action. Traced through, the expected end state is:

- document `1`: exists with `msg = "updated doc"` (indexed, then updated)
- document `2`: does not exist (created, then deleted)

The trailing newline matters. The Bulk API uses literal `\n` as its delimiter, so the final line must end with one.

In [13]:
os.makedirs(f"{WORK}/data", exist_ok=True)
bulk_lines = [
    '{ "index":  { "_index": "api-logs_2", "_id": "1" } }',
    '{ "msg": "indexed doc" }',
    '{ "create": { "_index": "api-logs_2", "_id": "2" } }',
    '{ "msg": "created doc" }',
    '{ "update": { "_index": "api-logs_2", "_id": "1" } }',
    '{ "doc": { "msg": "updated doc" } }',
    '{ "delete": { "_index": "api-logs_2", "_id": "2" } }',
]
bulk = "\n".join(bulk_lines) + "\n"  # required trailing newline
with open(f"{WORK}/data/bulk.ndjson", "w") as f:
    f.write(bulk)
print(bulk)


{ "index":  { "_index": "api-logs_2", "_id": "1" } }
{ "msg": "indexed doc" }
{ "create": { "_index": "api-logs_2", "_id": "2" } }
{ "msg": "created doc" }
{ "update": { "_index": "api-logs_2", "_id": "1" } }
{ "doc": { "msg": "updated doc" } }
{ "delete": { "_index": "api-logs_2", "_id": "2" } }



In [20]:
import uuid, requests
headers = {"Content-Type": "application/x-ndjson", "X-Request-Id": str(uuid.uuid4())}
with open(f"{WORK}/data/bulk.ndjson", "rb") as f:
    resp = requests.post(f"http://127.0.0.1:{HTTP_PORT}/", headers=headers, data=f.read(), timeout=10)
print("Logstash http input status:", resp.status_code)
print("Body:", resp.text or "(empty, the input acknowledges receipt)")

Logstash http input status: 200
Body: ok


## Step 4. Verify the operations in Elasticsearch

Indexing happens asynchronously in the Logstash output, so give it a moment, refresh the index, then read the documents back.

In [21]:
time.sleep(6)
requests.post("http://127.0.0.1:9200/api-logs_2/_refresh")

d1 = requests.get("http://127.0.0.1:9200/api-logs_2/_doc/1").json()
r2 = requests.get("http://127.0.0.1:9200/api-logs_2/_doc/2")
d2 = r2.json()

print("Document 1:", json.dumps(d1, indent=2))
print("Document 2 status:", r2.status_code)
print("Document 2:", json.dumps(d2, indent=2))
print()
print(requests.get("http://127.0.0.1:9200/_cat/indices/api-logs_2?v").text)

assert d1.get("found") and d1["_source"].get("msg") == "updated doc", d1
assert d2.get("found") is False, d2
print("Verified: index + update landed on doc 1, create + delete removed doc 2.")


Document 1: {
  "_index": "api-logs_2",
  "_id": "1",
  "_version": 2,
  "_seq_no": 2,
  "_primary_term": 1,
  "found": true,
  "_source": {
    "msg": "updated doc"
  }
}
Document 2 status: 404
Document 2: {
  "_index": "api-logs_2",
  "_id": "2",
  "found": false
}

health status index      uuid                   pri rep docs.count docs.deleted store.size pri.store.size dataset.size
yellow open   api-logs_2 5LvLAGH1R3C6CfL__OIzqA   1   1          1            3     29.9kb         29.9kb       29.9kb

Verified: index + update landed on doc 1, create + delete removed doc 2.


## Step 5. Inspect pipeline analytics with tuistash

[tuistash](https://github.com/edmocosta/tuistash) is a terminal UI for Logstash that reads the monitoring API (default `http://localhost:9600`). Its interactive mode (`tuistash tui`) needs a real terminal, which a notebook cell does not provide, so here the `get` subcommand is used for a non-interactive snapshot. Run the interactive TUI from your own terminal against a local Logstash.

For this pipeline the useful signals are events in versus events out (they will not match, because action lines are dropped after being queued) and the per-plugin counts on the aggregate and ruby filters.

In [22]:
import stat, tarfile

def install_tuistash():
    api = "https://api.github.com/repos/edmocosta/tuistash/releases/latest"
    data = json.load(urllib.request.urlopen(api))
    assets = data.get("assets", [])
    cand = None
    for a in assets:
        n = a["name"].lower()
        if "linux" in n and ("x86_64" in n or "amd64" in n):
            cand = a
            break
    if not cand:
        print("No prebuilt linux asset found. Available assets:", [a["name"] for a in assets])
        print("Build from source instead: see https://github.com/edmocosta/tuistash")
        return None
    fn = cand["name"]
    print("Downloading", cand["browser_download_url"])
    urllib.request.urlretrieve(cand["browser_download_url"], fn)
    binpath = None
    if fn.endswith(".tar.gz") or fn.endswith(".tgz"):
        with tarfile.open(fn) as t:
            t.extractall("tuistash_dist", filter="data")
        for root, _, files in os.walk("tuistash_dist"):
            for f in files:
                if f == "tuistash":
                    binpath = os.path.join(root, f)
    else:
        binpath = os.path.abspath(fn)
    if binpath:
        os.chmod(binpath, os.stat(binpath).st_mode | stat.S_IEXEC)
    return binpath

tuistash_bin = install_tuistash()
print("tuistash binary:", tuistash_bin)


tuistash binary: tuistash_dist/tuistash


In [23]:
# Non-interactive snapshot of node and pipeline stats
if tuistash_bin:
    out = subprocess.run(
        [tuistash_bin, "--host", "http://127.0.0.1:9600", "get", "node", "pipelines,os", "-o", "raw"],
        capture_output=True, text=True,
    )
    print(out.stdout or out.stderr)


{"host":"82fb42c5af44","version":"9.3.0","http_address":"127.0.0.1:9600","id":"eb6f0884-9528-4e80-9859-cfadb34f5006","name":"82fb42c5af44","ephemeral_id":"261472ba-8bfd-47eb-830b-2338b0145d81","snapshot":false,"status":"green","pipeline":{"workers":1,"batch_size":125,"batch_delay":50},"pipelines":{"main":{"ephemeral_id":"9c23c1ac-9d5c-4192-9509-3b8e59378ecb","hash":"4e24d136623039ada4886b80bb2a2cd79284998c4a0f644500d016954876ac21","workers":1,"batch_size":125,"batch_delay":50,"dead_letter_queue_enabled":false}},"os":{"name":"Linux","arch":"amd64","version":"6.6.122+","available_processors":2}}



In [24]:
# The raw monitoring API always works, with or without tuistash installed
stats = requests.get("http://127.0.0.1:9600/_node/stats/pipelines?pretty").text
print(stats[:4000])


{
  "host" : "82fb42c5af44",
  "version" : "9.3.0",
  "http_address" : "127.0.0.1:9600",
  "id" : "eb6f0884-9528-4e80-9859-cfadb34f5006",
  "name" : "82fb42c5af44",
  "ephemeral_id" : "261472ba-8bfd-47eb-830b-2338b0145d81",
  "snapshot" : false,
  "status" : "green",
  "pipeline" : {
    "workers" : 1,
    "batch_size" : 125,
    "batch_delay" : 50
  },
  "pipelines" : {
    "main" : {
      "events" : {
        "out" : 4,
        "duration_in_millis" : 3023,
        "queue_push_duration_in_millis" : 3,
        "filtered" : 7,
        "in" : 7
      },
      "flow" : {
        "queue_backpressure" : {
          "current" : 0.0,
          "last_1_minute" : 4.668E-5,
          "lifetime" : 4.118E-5
        },
        "worker_concurrency" : {
          "current" : 8.536E-4,
          "last_1_minute" : 0.04662,
          "lifetime" : 0.04149
        },
        "filter_throughput" : {
          "current" : 0.0,
          "last_1_minute" : 0.1089,
          "lifetime" : 0.09606
        },
  

## Cleanup (optional)

Stop the background services when you are done, or just end the Colab runtime.

In [ ]:
try:
    ls_proc.terminate()
except Exception as e:
    print("logstash:", e)
subprocess.run("pkill -f 'elasticsearch' 2>/dev/null; pkill -f 'logstash' 2>/dev/null; echo stopped", shell=True)


## References

- Elasticsearch Bulk API: https://www.elastic.co/docs/api/doc/elasticsearch/operation/operation-bulk
- Logstash http input plugin: https://www.elastic.co/docs/reference/logstash/plugins/plugins-inputs-http
- Logstash aggregate filter plugin: https://www.elastic.co/docs/reference/logstash/plugins/plugins-filters-aggregate
- Logstash Elasticsearch output plugin: https://www.elastic.co/docs/reference/logstash/plugins/plugins-outputs-elasticsearch
- Run Elasticsearch locally (start-local): https://www.elastic.co/docs/deploy-manage/deploy/self-managed/local-development-installation-quickstart
- tuistash: https://github.com/edmocosta/tuistash
- Discuss thread on forwarding bulk through Logstash: https://discuss.elastic.co/t/forwarding-the-http-input-data-to-http-output-data/375567
